In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 24.6 MB/s eta 0:00:00


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

print("GPU available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

GPU available: False
Device: cpu


In [2]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

pipe = pipeline(
    "text-generation",
    model=model_name,
    tokenizer=tokenizer,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
def chat(user_prompt, max_new_tokens=200, temperature=0.7, top_p=0.9):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": user_prompt},
    ]
    prompt = pipe.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    out = pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p
    )
    return out[0]["generated_text"]

In [5]:
print(chat("Explain what a small language model is in 3 bullet points."))

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'top_p', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|system|>
You are a helpful assistant.</s>
<|user|>
Explain what a small language model is in 3 bullet points.</s>
<|assistant|>
1. Small Language Model (SLM): A small language model is a type of model used in language translation tasks. It is a type of artificial neural network that can learn a translation process from training data. SLMs are typically trained using machine translation data, which consists of input and output texts. The size of SLMs is usually less than 10,000 parameters.

2. Small Language Model (SLS): SLS is a variation of SLMs. SLS uses a subset of parameters from the parent model, which can be trained on a smaller set of training data. SLS is typically used for tasks where the training data is not as large as for SLMs.

3. Small Language Model (SLSL): SLSL is a small language model that is trained on a subset of parameters from a larger SLM. SLSL uses a subset of the parent SLM's parameters, but only a subset of


In [6]:
print(chat("what is nature in 3 bullet points."))

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|system|>
You are a helpful assistant.</s>
<|user|>
what is nature in 3 bullet points.</s>
<|assistant|>
1. Nature is a source of inspiration and relaxation for many people.
2. It provides a sense of calm and relaxation.
3. Nature can be a source of solace and comfort during difficult times.
